<a href="https://colab.research.google.com/github/AsimaZaheer/Stress-Detection/blob/main/Stress_Detection_4Hz_Resampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
WESAD Stress Detection Pipeline
================================
Dataset Google Drive se automatically download hoga.
User ko apni Google Drive mount karne ki zarurat nahi.

IMPORTANT:
Sirf WESAD_ZIP_URL mein apna public Google Drive ZIP link paste karein.
"""

# ============================================================
# IMPORTS
# ============================================================

import os
import pickle
import zipfile
import numpy as np

from scipy import signal
from scipy.signal import find_peaks

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# DATASET DOWNLOAD CONFIGURATION
# ============================================================

# ============================================================
# IMPORTANT:
# YAHAN APNA GOOGLE DRIVE WESAD.ZIP LINK PASTE KAREIN
# Example:
#
# WESAD_ZIP_URL = "https://drive.google.com/file/d/XXXXXXXX/view?usp=sharing"
# ============================================================

WESAD_ZIP_URL = "https://drive.google.com/file/d/1MbfU2z4OnyevB0oX_yvHVue7Wv24KP7l/view?usp=sharing"


# Temporary location in Google Colab
ZIP_PATH = "/content/WESAD.zip"

# Dataset extraction location
EXTRACT_PATH = "/content"


# ============================================================
# DOWNLOAD DATASET FROM GOOGLE DRIVE
# ============================================================

def download_wesad_dataset():

    print("=" * 60)
    print("WESAD DATASET SETUP")
    print("=" * 60)

    if WESAD_ZIP_URL == "PASTE_YOUR_GOOGLE_DRIVE_ZIP_LINK_HERE":
        raise ValueError(
            "\nPlease paste your Google Drive WESAD ZIP link "
            "inside WESAD_ZIP_URL."
        )

    # Install gdown
    print("\n📦 Installing gdown...")
    os.system("pip -q install gdown")

    import gdown

    # Download ZIP
    print("\n⬇️ Downloading WESAD dataset...")
    print("This may take some time depending on internet speed.\n")

    gdown.download(
        WESAD_ZIP_URL,
        ZIP_PATH,
        quiet=False,
        fuzzy=True
    )

    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(
            "WESAD ZIP file could not be downloaded."
        )

    print("\n✅ WESAD ZIP downloaded successfully.")

    # Extract ZIP
    print("\n📂 Extracting WESAD dataset...")

    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

    print("✅ Dataset extracted successfully.")

    # Show extracted folders
    print("\n📁 Checking extracted dataset structure...")

    possible_paths = [
        "/content/WESAD",
        "/content/WESAD/WESAD",
    ]

    dataset_path = None

    for path in possible_paths:
        if os.path.isdir(path):
            dataset_path = path
            break

    if dataset_path is None:

        # Search automatically for a folder containing S2
        for root, dirs, files in os.walk("/content"):

            if "S2" in dirs:
                candidate = os.path.join(root)

                s2_path = os.path.join(candidate, "S2")

                if os.path.isdir(s2_path):
                    dataset_path = candidate
                    break

    if dataset_path is None:
        raise FileNotFoundError(
            """
WESAD dataset folder could not be located.

Please make sure your ZIP contains:
WESAD/
    S2/
    S3/
    S4/
    ...
"""
        )

    print(f"\n✅ Dataset location found:")
    print(dataset_path)

    return dataset_path


# ============================================================
# AUTOMATICALLY DOWNLOAD DATASET
# ============================================================

DATASET_PATH = download_wesad_dataset()


# ============================================================
# CONFIG
# ============================================================

SUBJECTS = [
    f"S{i}"
    for i in [
        2, 3, 4, 5, 6, 7, 8,
        9, 10, 11, 13, 14, 15, 16, 17
    ]
]

TARGET_SAMPLING_RATE = 4
LABEL_NATIVE_RATE = 700

WINDOW_SIZE = 60
STEP_SIZE = 10

BATCH_SIZE = 32

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("\n🖥️ Device:", DEVICE)
print("📁 Dataset:", DATASET_PATH)
print("👥 Subjects:", SUBJECTS)


# ============================================================
# HELPER: RESAMPLING
# ============================================================

def resample_signal(sig, orig_rate, target_rate):

    n_samples = int(
        len(sig) * target_rate / orig_rate
    )

    return signal.resample(
        sig,
        n_samples
    )


# ============================================================
# LABEL ALIGNMENT
# ============================================================

def align_labels_by_time(
    labels_raw,
    label_rate,
    n_target_samples,
    target_rate
):

    target_times = (
        np.arange(n_target_samples)
        / target_rate
    )

    label_indices = np.round(
        target_times * label_rate
    ).astype(int)

    label_indices = np.clip(
        label_indices,
        0,
        len(labels_raw) - 1
    )

    return labels_raw[label_indices]


# ============================================================
# HRV FEATURES
# ============================================================

def extract_hrv_features(
    bvp_window,
    fs
):

    peaks, _ = find_peaks(
        bvp_window,
        distance=max(
            1,
            int(fs * 0.4)
        ),
        prominence=0.1
    )

    if len(peaks) < 3:
        return np.array([
            np.nan,
            np.nan,
            np.nan
        ])

    rr_intervals = (
        np.diff(peaks) / fs
    )

    hr = 60.0 / rr_intervals

    mean_hr = np.mean(hr)

    sdnn = (
        np.std(rr_intervals)
        * 1000.0
    )

    rmssd = (
        np.sqrt(
            np.mean(
                np.diff(rr_intervals) ** 2
            )
        )
        * 1000.0
    )

    return np.array([
        mean_hr,
        sdnn,
        rmssd
    ])


# ============================================================
# WINDOWING + HRV FEATURES
# ============================================================

def create_windows_with_features(
    eda,
    bvp,
    labels,
    fs,
    window_size_s,
    step_size_s
):

    window_samples = int(
        window_size_s * fs
    )

    step_samples = int(
        step_size_s * fs
    )

    X_seq = []
    X_hrv = []
    y = []

    for i in range(
        0,
        len(eda) - window_samples,
        step_samples
    ):

        eda_win = eda[
            i:i + window_samples
        ]

        bvp_win = bvp[
            i:i + window_samples
        ]

        label_win = labels[
            i:i + window_samples
        ]

        unique_labels, counts = np.unique(
            label_win,
            return_counts=True
        )

        majority_label = unique_labels[
            np.argmax(counts)
        ]

        # Valid WESAD labels:
        # 1 = Baseline
        # 2 = Stress
        # 3 = Amusement

        if majority_label in [1, 2, 3]:

            hrv_feats = extract_hrv_features(
                bvp_win,
                fs
            )

            if np.isnan(hrv_feats).any():
                continue

            # EDA + BVP
            seq_window = np.column_stack(
                (
                    eda_win,
                    bvp_win
                )
            )

            X_seq.append(seq_window)
            X_hrv.append(hrv_feats)

            # Convert:
            # 1 -> 0
            # 2 -> 1
            # 3 -> 2

            y.append(
                majority_label - 1
            )

    return (
        np.array(X_seq),
        np.array(X_hrv),
        np.array(y)
    )


# ============================================================
# CNN + HRV MODEL
# ============================================================

class StressClassifier(nn.Module):

    def __init__(
        self,
        seq_channels=2,
        hrv_dim=3,
        n_classes=3
    ):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv1d(
                seq_channels,
                16,
                kernel_size=5,
                padding=2
            ),

            nn.ReLU(),

            nn.MaxPool1d(2),

            nn.Conv1d(
                16,
                32,
                kernel_size=5,
                padding=2
            ),

            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                32 + hrv_dim,
                32
            ),

            nn.ReLU(),

            nn.Linear(
                32,
                n_classes
            )
        )

    def forward(
        self,
        x_seq,
        x_hrv
    ):

        # Input:
        # (batch, window_length, channels)

        x = x_seq.permute(
            0,
            2,
            1
        )

        x = self.conv(x)

        x = x.squeeze(-1)

        # Add HRV features

        x = torch.cat(
            [
                x,
                x_hrv
            ],
            dim=1
        )

        return self.classifier(x)


# ============================================================
# WESAD PIPELINE
# ============================================================

class WESADPipeline:

    def __init__(
        self,
        dataset_path,
        subjects,
        fs=TARGET_SAMPLING_RATE,
        label_rate=LABEL_NATIVE_RATE,
        window_size=WINDOW_SIZE,
        step_size=STEP_SIZE
    ):

        self.dataset_path = dataset_path
        self.subjects = subjects
        self.fs = fs
        self.label_rate = label_rate
        self.window_size = window_size
        self.step_size = step_size


    # ========================================================
    # LOAD SUBJECT
    # ========================================================

    def load_subject_raw(
        self,
        sub
    ):

        file_path = os.path.join(
            self.dataset_path,
            sub,
            f"{sub}.pkl"
        )

        if not os.path.exists(file_path):

            print(
                f"⚠️ {file_path} not found."
            )

            return None

        print(
            f"Loading {sub}..."
        )

        with open(
            file_path,
            "rb"
        ) as f:

            data = pickle.load(
                f,
                encoding="latin1"
            )

        # ------------------------------
        # Extract signals
        # ------------------------------

        eda_raw = data[
            "signal"
        ][
            "wrist"
        ][
            "EDA"
        ].flatten()

        bvp_raw = data[
            "signal"
        ][
            "wrist"
        ][
            "BVP"
        ].flatten()

        labels_raw = data[
            "label"
        ].flatten()

        # ------------------------------
        # Resample
        # ------------------------------

        eda_resampled = resample_signal(
            eda_raw,
            4,
            self.fs
        )

        bvp_resampled = resample_signal(
            bvp_raw,
            64,
            self.fs
        )

        # ------------------------------
        # Same length
        # ------------------------------

        n_target = min(
            len(eda_resampled),
            len(bvp_resampled)
        )

        eda_resampled = (
            eda_resampled[:n_target]
        )

        bvp_resampled = (
            bvp_resampled[:n_target]
        )

        # ------------------------------
        # Align labels
        # ------------------------------

        labels_aligned = (
            align_labels_by_time(
                labels_raw,
                self.label_rate,
                n_target,
                self.fs
            )
        )

        return (
            eda_resampled,
            bvp_resampled,
            labels_aligned
        )


    # ========================================================
    # BUILD WINDOWS FOR ALL SUBJECTS
    # ========================================================

    def build_all_subject_windows(self):

        X_seq_list = []
        X_hrv_list = []
        y_list = []
        groups_list = []

        for sub in self.subjects:

            loaded = self.load_subject_raw(
                sub
            )

            if loaded is None:
                continue

            eda, bvp, labels = loaded

            X_seq, X_hrv, y = (
                create_windows_with_features(
                    eda,
                    bvp,
                    labels,
                    self.fs,
                    self.window_size,
                    self.step_size
                )
            )

            if len(y) == 0:

                print(
                    f"⚠️ {sub}: No valid windows."
                )

                continue

            X_seq_list.append(X_seq)
            X_hrv_list.append(X_hrv)
            y_list.append(y)

            groups_list.append(
                np.full(
                    len(y),
                    sub
                )
            )

            print(
                f"  {sub}: {len(y)} windows"
            )

        # ------------------------------
        # Combine all subjects
        # ------------------------------

        X_seq_all = np.concatenate(
            X_seq_list,
            axis=0
        )

        X_hrv_all = np.concatenate(
            X_hrv_list,
            axis=0
        )

        y_all = np.concatenate(
            y_list,
            axis=0
        )

        groups_all = np.concatenate(
            groups_list,
            axis=0
        )

        return (
            X_seq_all,
            X_hrv_all,
            y_all,
            groups_all
        )


    # ========================================================
    # LOSO CROSS VALIDATION
    # ========================================================

    def run_loso(
        self,
        epochs=15,
        lr=1e-3,
        device=DEVICE
    ):

        print(
            "\n⏳ Loading and processing all subjects..."
        )

        (
            X_seq_all,
            X_hrv_all,
            y_all,
            groups_all
        ) = self.build_all_subject_windows()

        # ------------------------------
        # Class distribution
        # ------------------------------

        unique, counts = np.unique(
            y_all,
            return_counts=True
        )

        print(
            "\n📊 Overall class distribution:"
        )

        print(
            dict(
                zip(
                    unique,
                    counts
                )
            )
        )

        # ------------------------------
        # LOSO
        # ------------------------------

        logo = LeaveOneGroupOut()

        fold_reports = []

        total_folds = len(
            np.unique(groups_all)
        )

        for fold_idx, (
            train_idx,
            test_idx
        ) in enumerate(
            logo.split(
                X_seq_all,
                y_all,
                groups_all
            )
        ):

            test_subject = (
                groups_all[test_idx][0]
            )

            print(
                "\n"
                + "=" * 60
            )

            print(
                f"FOLD {fold_idx + 1}/{total_folds}"
            )

            print(
                f"Test Subject: {test_subject}"
            )

            print(
                "=" * 60
            )

            # --------------------------
            # Train/Test split
            # --------------------------

            X_seq_train = (
                X_seq_all[train_idx]
            )

            X_seq_test = (
                X_seq_all[test_idx]
            )

            X_hrv_train = (
                X_hrv_all[train_idx]
            )

            X_hrv_test = (
                X_hrv_all[test_idx]
            )

            y_train = (
                y_all[train_idx]
            )

            y_test = (
                y_all[test_idx]
            )

            # --------------------------
            # Sequence scaling
            # --------------------------

            n_train, wlen, n_ch = (
                X_seq_train.shape
            )

            n_test = (
                X_seq_test.shape[0]
            )

            seq_scaler = StandardScaler()

            X_seq_train_s = (
                seq_scaler.fit_transform(
                    X_seq_train.reshape(
                        -1,
                        n_ch
                    )
                ).reshape(
                    n_train,
                    wlen,
                    n_ch
                )
            )

            X_seq_test_s = (
                seq_scaler.transform(
                    X_seq_test.reshape(
                        -1,
                        n_ch
                    )
                ).reshape(
                    n_test,
                    wlen,
                    n_ch
                )
            )

            # --------------------------
            # HRV scaling
            # --------------------------

            hrv_scaler = StandardScaler()

            X_hrv_train_s = (
                hrv_scaler.fit_transform(
                    X_hrv_train
                )
            )

            X_hrv_test_s = (
                hrv_scaler.transform(
                    X_hrv_test
                )
            )

            # --------------------------
            # Class weights
            # --------------------------

            class_counts_train = (
                np.bincount(
                    y_train,
                    minlength=3
                ).astype(
                    np.float32
                )
            )

            class_weights = (
                class_counts_train.sum()
                /
                (
                    len(class_counts_train)
                    *
                    np.clip(
                        class_counts_train,
                        1,
                        None
                    )
                )
            )

            class_weights_t = torch.tensor(
                class_weights,
                dtype=torch.float32,
                device=device
            )

            print(
                "Train class counts:",
                dict(
                    zip(
                        [
                            "Baseline",
                            "Stress",
                            "Amusement"
                        ],
                        class_counts_train.astype(
                            int
                        )
                    )
                )
            )

            print(
                "Class weights:",
                np.round(
                    class_weights,
                    3
                )
            )

            # --------------------------
            # Tensor datasets
            # --------------------------

            train_ds = TensorDataset(

                torch.tensor(
                    X_seq_train_s,
                    dtype=torch.float32
                ),

                torch.tensor(
                    X_hrv_train_s,
                    dtype=torch.float32
                ),

                torch.tensor(
                    y_train,
                    dtype=torch.long
                )
            )

            test_ds = TensorDataset(

                torch.tensor(
                    X_seq_test_s,
                    dtype=torch.float32
                ),

                torch.tensor(
                    X_hrv_test_s,
                    dtype=torch.float32
                ),

                torch.tensor(
                    y_test,
                    dtype=torch.long
                )
            )

            train_loader = DataLoader(
                train_ds,
                batch_size=BATCH_SIZE,
                shuffle=True
            )

            test_loader = DataLoader(
                test_ds,
                batch_size=BATCH_SIZE,
                shuffle=False
            )

            # --------------------------
            # Model
            # --------------------------

            model = StressClassifier().to(
                device
            )

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=lr
            )

            criterion = (
                nn.CrossEntropyLoss(
                    weight=class_weights_t
                )
            )

            # --------------------------
            # Training
            # --------------------------

            model.train()

            for epoch in range(
                epochs
            ):

                total_loss = 0.0

                for (
                    xb_seq,
                    xb_hrv,
                    yb
                ) in train_loader:

                    xb_seq = (
                        xb_seq.to(device)
                    )

                    xb_hrv = (
                        xb_hrv.to(device)
                    )

                    yb = (
                        yb.to(device)
                    )

                    optimizer.zero_grad()

                    output = model(
                        xb_seq,
                        xb_hrv
                    )

                    loss = criterion(
                        output,
                        yb
                    )

                    loss.backward()

                    optimizer.step()

                    total_loss += (
                        loss.item()
                    )

                if (
                    (epoch + 1) % 5 == 0
                    or epoch == epochs - 1
                ):

                    avg_loss = (
                        total_loss
                        /
                        len(train_loader)
                    )

                    print(
                        f"Epoch "
                        f"{epoch + 1}/{epochs}"
                        f" - Loss: "
                        f"{avg_loss:.4f}"
                    )

            # --------------------------
            # Evaluation
            # --------------------------

            model.eval()

            all_preds = []
            all_true = []

            with torch.no_grad():

                for (
                    xb_seq,
                    xb_hrv,
                    yb
                ) in test_loader:

                    xb_seq = (
                        xb_seq.to(device)
                    )

                    xb_hrv = (
                        xb_hrv.to(device)
                    )

                    output = model(
                        xb_seq,
                        xb_hrv
                    )

                    preds = (
                        output
                        .argmax(dim=1)
                        .cpu()
                        .numpy()
                    )

                    all_preds.extend(
                        preds
                    )

                    all_true.extend(
                        yb.numpy()
                    )

            # --------------------------
            # Accuracy
            # --------------------------

            acc = accuracy_score(
                all_true,
                all_preds
            )

            print(
                f"\n✅ {test_subject} "
                f"Accuracy: {acc:.3f}"
            )

            print(
                classification_report(
                    all_true,
                    all_preds,
                    labels=[0, 1, 2],
                    target_names=[
                        "Baseline",
                        "Stress",
                        "Amusement"
                    ],
                    zero_division=0
                )
            )

            fold_reports.append({

                "test_subject":
                    test_subject,

                "accuracy":
                    acc,

                "n_train":
                    n_train,

                "n_test":
                    n_test
            })

        return fold_reports


# ============================================================
# RUN PIPELINE
# ============================================================

if __name__ == "__main__":

    print(
        "\n🚀 Starting WESAD Stress Detection..."
    )

    pipeline = WESADPipeline(
        DATASET_PATH,
        SUBJECTS
    )

    results = pipeline.run_loso(
        epochs=15,
        lr=1e-3
    )

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    accs = [
        r["accuracy"]
        for r in results
    ]

    print(
        "\n"
        + "=" * 60
    )

    print(
        "LOSO SUMMARY"
    )

    print(
        "=" * 60
    )

    for r in results:

        print(
            f"{r['test_subject']}: "
            f"Accuracy={r['accuracy']:.3f} "
            f"(train={r['n_train']}, "
            f"test={r['n_test']})"
        )

    print(
        "\nMean LOSO Accuracy: "
        f"{np.mean(accs):.3f}"
    )

    print(
        "Standard Deviation: "
        f"{np.std(accs):.3f}"
    )

    print(
        "\n🎉 WESAD pipeline completed."
    )

WESAD DATASET SETUP

📦 Installing gdown...

⬇️ Downloading WESAD dataset...
This may take some time depending on internet speed.



Downloading...
From (original): https://drive.google.com/uc?id=1MbfU2z4OnyevB0oX_yvHVue7Wv24KP7l
From (redirected): https://drive.google.com/uc?id=1MbfU2z4OnyevB0oX_yvHVue7Wv24KP7l&confirm=t&uuid=06406664-55cd-49b0-a2b1-b742d27b9de0
To: /content/WESAD.zip
100%|██████████| 2.25G/2.25G [00:24<00:00, 92.3MB/s]



✅ WESAD ZIP downloaded successfully.

📂 Extracting WESAD dataset...
✅ Dataset extracted successfully.

📁 Checking extracted dataset structure...

✅ Dataset location found:
/content/WESAD

🖥️ Device: cpu
📁 Dataset: /content/WESAD
👥 Subjects: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']

🚀 Starting WESAD Stress Detection...

⏳ Loading and processing all subjects...
Loading S2...
  S2: 212 windows
Loading S3...
  S3: 215 windows
Loading S4...
  S4: 216 windows
Loading S5...
  S5: 221 windows
Loading S6...
  S6: 220 windows
Loading S7...
  S7: 219 windows
Loading S8...
  S8: 221 windows
Loading S9...
  S9: 220 windows
Loading S10...
  S10: 227 windows
Loading S11...
  S11: 223 windows
Loading S13...
  S13: 222 windows
Loading S14...
  S14: 222 windows
Loading S15...
  S15: 223 windows
Loading S16...
  S16: 222 windows
Loading S17...
  S17: 227 windows

📊 Overall class distribution:
{np.int32(0): np.int64(1761), np.int32(1): np.int64(994